In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
import os
import numpy as np
pd.set_option('display.max_columns', 1000);

Dashboard Objectives

    Provide interactive visualizations for key patient health indicators.
    Assist nursing students in understanding clinical assessments from OASIS and county level data (Census & SDOH).


In [3]:
import plotly.express as px
import plotly.graph_objects as go
import holoviews as hv
from bokeh.palettes import Category10
import scipy.cluster.hierarchy as sch

from bokeh.plotting import figure

In [4]:
import panel as pn
pn.extension()


In [5]:
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

counties["features"][0]

{'type': 'Feature',
 'properties': {'GEO_ID': '0500000US01001',
  'STATE': '01',
  'COUNTY': '001',
  'NAME': 'Autauga',
  'LSAD': 'County',
  'CENSUSAREA': 594.436},
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-86.496774, 32.344437],
    [-86.717897, 32.402814],
    [-86.814912, 32.340803],
    [-86.890581, 32.502974],
    [-86.917595, 32.664169],
    [-86.71339, 32.661732],
    [-86.714219, 32.705694],
    [-86.413116, 32.707386],
    [-86.411172, 32.409937],
    [-86.496774, 32.344437]]]},
 'id': '01001'}

In [6]:
df = pd.read_csv('data/dashboardData_ICDSections_v2.csv', low_memory=False)

In [7]:
df

,Beneficiary_ID,Assessment_Effective_Date,Submitted_HIPPS_Code,Facility_Internal_ID,Age,Gender,American_Indian_or_Alaska_Native,Asian,Black_or_African_American,Hispanic_or_Latino,Native_Hawiian_or_Pacific_Islander,White,ByDiscipline,Days_Cared_For,Agency_Medicare_Number,Primary_Diagnosis_ICD_10_C_M_Code,Other_Diagnosis_Code_1_ICD_10_C_M,Other_Diagnosis_Code_2_ICD_10_C_M,Other_Diagnosis_Code_3_ICD_10_C_M,Other_Diagnosis_Code_4_ICD_10_C_M,Other_Diagnosis_Code_5_ICD_10_C_M,has_diabetes,has_heart_failure,has_hypertension,ever_deceased,ever_readmitted,BMI,BMI_Category,COUNTY_NAME,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,COUNTYFIPS,ACS_PCT_BACHELOR_DGR,ACS_PCT_COLLEGE_ASSOCIATE_DGR,ACS_PCT_GRADUATE_DGR,ACS_PCT_HS_GRADUATE,ACS_PCT_LT_HS,ACS_PCT_NO_WORK_NO_SCHL_16_19,ACS_PCT_POSTHS_ED,ACS_PCT_VET_BACHELOR,ACS_PCT_VET_COLLEGE,ACS_PCT_VET_HS,ACS_PCT_HH_LIMIT_ENGLISH,ACS_PCT_HH_BROADBAND,ACS_PCT_HH_BROADBAND_ONLY,ACS_PCT_HH_CELLULAR,ACS_PCT_HH_CELLULAR_ONLY,ACS_PCT_HH_DIAL_INTERNET_ONLY,ACS_PCT_HH_INTERNET,ACS_PCT_HH_INTERNET_NO_SUBS,ACS_PCT_HH_NO_COMP_DEV,ACS_PCT_HH_NO_INTERNET,ACS_PCT_HH_OTHER_COMP,ACS_PCT_HH_OTHER_COMP_ONLY,ACS_PCT_HH_PC,ACS_PCT_HH_PC_ONLY,ACS_PCT_HH_SAT_INTERNET,ACS_PCT_HH_SMARTPHONE,ACS_PCT_HH_SMARTPHONE_ONLY,ACS_PCT_HH_TABLET,ACS_PCT_HH_TABLET_ONLY,ACS_PCT_CHILDREN_GRANDPARENT,ACS_PCT_CHILD_1FAM,ACS_PCT_GRANDP_NO_RESPS,ACS_PCT_GRANDP_RESPS_NO_P,ACS_PCT_GRANDP_RESPS_P,ACS_PCT_HH_1PERS,ACS_PCT_HH_ABOVE65,ACS_PCT_HH_ALONE_ABOVE65,ACS_PCT_HH_KID_1PRNT,ACS_TOT_GRANDCHILDREN_GP,ACS_PCT_HEALTH_INC_138_199,ACS_PCT_HEALTH_INC_200_399,ACS_PCT_HEALTH_INC_ABOVE400,ACS_PCT_HEALTH_INC_BELOW137,ACS_PCT_HH_1FAM_FOOD_STMP,ACS_PCT_HH_FOOD_STMP_BLW_POV,ACS_PCT_HH_NO_FD_STMP_BLW_POV,ACS_PCT_HH_PUB_ASSIST,ACS_PCT_INC50,ACS_PCT_INC50_ABOVE65,ACS_PCT_NONVET_POV_18_64,ACS_PCT_PERSON_INC_100_124,ACS_PCT_PERSON_INC_125_199,ACS_PCT_PERSON_INC_ABOVE200,ACS_PCT_PERSON_INC_BELOW99,ACS_PCT_POV_AIAN,ACS_PCT_POV_ASIAN,ACS_PCT_POV_BLACK,ACS_PCT_POV_HISPANIC,ACS_PCT_POV_MULTI,ACS_PCT_POV_NHPI,ACS_PCT_POV_OTHER,ACS_PCT_POV_WHITE,ACS_PCT_VET_POV_18_64,ACS_TOT_POP_POV,ICD_Section,ICD_Range
0,GGGGGGG99DzGeze,2017-03-17,1AGKU,334582,46,Female,0,0,1,0,0,0,RN,25,677994,C25.7,R10.11,N18.5,D64.9,Z45.2,M62.81,0,0,0,1,1,29.898299,Overweight,Harris,4677507.0,98.87,53638.0,1.13,48201,20.002284,25.887536,11.996951,22.838174,19.185522,1.939011,57.886619,31.882239,36.757860,28.122410,12.653121,68.342320,6.312617,78.048615,14.276799,0.141951,85.394164,2.052950,7.455809,12.283121,2.412104,0.018714,74.575108,3.227824,7.685683,86.899218,12.036313,59.834092,0.734829,8.129937,33.536888,38.805908,10.236151,24.519155,26.830728,21.947275,7.247464,21.881277,93.839029,12.250612,28.372572,34.331007,24.776376,24.694119,6.257833,9.230647,14.404757,7.291655,4.506214,14.438022,5.353849,14.852644,62.705198,16.818489,6.254451,11.774038,17.916232,18.082023,14.639892,0.612113,17.751070,13.141808,9.128750,4167.452338,{'Diseases of the musculoskeletal system and c...,"{'M00-M99', 'D50-D89', 'Z00-Z99', 'N00-N99', '..."
1,GGGGGGG9DDWWeeX,2017-10-22,1CHKS,63847,75,Male,0,0,0,0,0,1,RN,37,679236,Z48.812,R55.,I44.69,M62.81,I25.10,F03.90,0,0,0,1,0,27.194009,Overweight,Collin,999005.0,93.85,65460.0,6.15,48085,33.544045,26.076091,19.293682,14.445273,6.186000,0.708045,78.914227,47.769182,35.078682,15.788318,4.921545,83.973455,8.104000,84.431545,6.473273,0.072636,93.578818,1.297773,1.709727,4.669455,4.171000,0.042864,90.940864,2.370727,7.718545,93.952591,4.400864,74.761636,0.326227,3.902545,20.265091,34.616136,12.416045,20.240682,21.790864,21.080136,6.321000,12.976727,45.881818,6.623636,23.725682,58.459455,10.736455,7.268955,0.904455,5.726818,3.434955,3.960545,4.068682,6.547545,2.374818,7.981318,82.184273,7.004545,4.326727,8.702818,8.223818,8.853182,7.923045,1.074364,7.404455,5.317909,4.171409,4546.331818,"{'Diseases of the circulatory system', 'Behavi...","{'M00-M99', 'F01-F99', 'Z00-Z99', 'I00-I99', '..."
2,GGGGGGG9DGXzUSG,2017-09-13,1BHKT,10206,75,Male,0,0,0,0,0,1,RN,57,677247,G61.81,Z45.2,M62.81,

In [8]:
df['COUNTYFIPS'] = df['COUNTYFIPS'].astype(int)

In [9]:
df.columns.tolist()

['Beneficiary_ID',
 'Assessment_Effective_Date',
 'Submitted_HIPPS_Code',
 'Facility_Internal_ID',
 'Age',
 'Gender',
 'American_Indian_or_Alaska_Native',
 'Asian',
 'Black_or_African_American',
 'Hispanic_or_Latino',
 'Native_Hawiian_or_Pacific_Islander',
 'White',
 'ByDiscipline',
 'Days_Cared_For',
 'Agency_Medicare_Number',
 'Primary_Diagnosis_ICD_10_C_M_Code',
 'Other_Diagnosis_Code_1_ICD_10_C_M',
 'Other_Diagnosis_Code_2_ICD_10_C_M',
 'Other_Diagnosis_Code_3_ICD_10_C_M',
 'Other_Diagnosis_Code_4_ICD_10_C_M',
 'Other_Diagnosis_Code_5_ICD_10_C_M',
 'has_diabetes',
 'has_heart_failure',
 'has_hypertension',
 'ever_deceased',
 'ever_readmitted',
 'BMI',
 'BMI_Category',
 'COUNTY_NAME',
 'POP_URB',
 'POPPCT_URB',
 'POP_RUR',
 'POPPCT_RUR',
 'COUNTYFIPS',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
 'ACS_PCT_GRADUATE_DGR',
 'ACS_PCT_HS_GRADUATE',
 'ACS_PCT_LT_HS',
 'ACS_PCT_NO_WORK_NO_SCHL_16_19',
 'ACS_PCT_POSTHS_ED',
 'ACS_PCT_VET_BACHELOR',
 'ACS_PCT_VET_COLLEGE',
 'A

In [10]:
from shapely.geometry import shape
import geopandas as gpd

In [11]:
features = counties["features"]
geometry = [shape(feature['geometry']) for feature in features]
names = [feature['properties']['NAME'] for feature in features]
fips = [feature['id'] for feature in features]
states = [feature['properties']['STATE'] for feature in features]

geo_df = gpd.GeoDataFrame({
    'County': names,
    'Fips': fips,
    'geometry': geometry,
    'State': states
})
geo_df

,County,Fips,geometry,State
0,Autauga,01001,"POLYGON ((-86.49677 32.34444, -86.7179 32.4028...",01
1,Blount,01009,"POLYGON ((-86.5778 33.76532, -86.75914 33.8406...",01
2,Chambers,01017,"POLYGON ((-85.18413 32.87052, -85.12342 32.772...",01
3,Chilton,01021,"POLYGON ((-86.51734 33.02057, -86.51596 32.929...",01
4,Colbert,01033,"POLYGON ((-88.13999 34.5817, -88.13925 34.5878...",01
...,...,...,...,...
3216,Accomack,51001,"MULTIPOLYGON (((-75.24227 38.02721, -75.29687 ...",51
3217,Bland,51021,"POLYGON ((-81.2251 37.23487, -81.20477 37.2430...",51
3218,Buchanan,51027,"POLYGON ((-81.9683 37.5378, -81.92787 37.51212...",51
3219,Charlotte,51037,"POLYGON ((-78.44332 37.0794, -78.49303 36.8912...",51


In [12]:
geo_df.set_crs('EPSG:4326', allow_override=True, inplace=True)
#source: https://public-rma.fpac.usda.gov/ReportShare//Extranet/SOB/Listings/StateAbbr2014.pdf
data = {
    "State_Abbreviation": [
        "AK", "AL", "AR", "AS", "AS", "AZ", "CA", "CO", "CT", "DC", "DE", "FL", "FM", "GA", "GU", "HI", "IA", "ID", "IL", "IN", "KS", "KY", "LA", 
        "MA", "MD", "ME", "MH", "MI", "MN", "MO", "MP", "MS", "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK", "OR", "PA", "PR", 
        "PR", "PW", "RI", "SC", "SD", "TN", "TX", "UM", "UT", "VA", "VI", "VI", "VT", "WA", "WI", "WV", "WY", "ZZ"
    ],
    "State_name": [
        "ALASKA", "ALABAMA", "ARKANSAS", "AMERICAN SAMOA", "ALL STATES", "ARIZONA", "CALIFORNIA", "COLORADO", "CONNECTICUT", "D. OF COLUMBIA",
        "DELAWARE", "FLORIDA", "FS OF MICRONESI", "GEORGIA", "GUAM", "HAWAII", "IOWA", "IDAHO", "ILLINOIS", "INDIANA", "KANSAS", "KENTUCKY",
        "LOUISIANA", "MASSACHUSETTS", "MARYLAND", "MAINE", "MARSHALL ISLAND", "MICHIGAN", "MINNESOTA", "MISSOURI", "NORTHERN MARIAN", 
        "MISSISSIPPI", "MONTANA", "NORTH CAROLINA", "NORTH DAKOTA", "NEBRASKA", "NEW HAMPSHIRE", "NEW JERSEY", "NEW MEXICO", "NEVADA", 
        "NEW YORK", "OHIO", "OKLAHOMA", "OREGON", "PENNSYLVANIA", "PUERTO RICO", "PUERTO RICO", "PALAU", "RHODE ISLAND", "SOUTH CAROLINA", 
        "SOUTH DAKOTA", "TENNESSEE", "TEXAS", "U.S. MINOR OUTL", "UTAH", "VIRGINIA", "VIRGIN ISLANDS", "VIRGIN ISLANDS", "VERMONT", "WASHINGTON", 
        "WISCONSIN", "WEST VIRGINIA", "WYOMING", "FOREIGN"
    ],
    "Code": [
        "02", "01", "05", "60", "98", "04", "06", "08", "09", "11", "10", "12", "64", "13", "66", "15", "19", "16", "17", "18", "20", "21", "22", 
        "25", "24", "23", "68", "26", "27", "29", "69", "28", "30", "37", "38", "31", "33", "34", "35", "32", "36", "39", "40", "41", "42", "72", 
        "43", "70", "44", "45", "46", "47", "48", "74", "49", "51", "78", "52", "50", "53", "55", "54", "56", "99"
    ]
}

codes = pd.DataFrame(data)
codes

,State_Abbreviation,State_name,Code
0,AK,ALASKA,02
1,AL,ALABAMA,01
2,AR,ARKANSAS,05
3,AS,AMERICAN SAMOA,60
4,AS,ALL STATES,98
...,...,...,...
59,WA,WASHINGTON,53
60,WI,WISCONSIN,55
61,WV,WEST VIRGINIA,54
62,WY,WYOMING,56


In [13]:
merged_data = pd.merge(geo_df, codes, right_on=['Code'], left_on=['State'])
merged_data

,County,Fips,geometry,State,State_Abbreviation,State_name,Code
0,Autauga,01001,"POLYGON ((-86.49677 32.34444, -86.7179 32.4028...",01,AL,ALABAMA,01
1,Blount,01009,"POLYGON ((-86.5778 33.76532, -86.75914 33.8406...",01,AL,ALABAMA,01
2,Chambers,01017,"POLYGON ((-85.18413 32.87052, -85.12342 32.772...",01,AL,ALABAMA,01
3,Chilton,01021,"POLYGON ((-86.51734 33.02057, -86.51596 32.929...",01,AL,ALABAMA,01
4,Colbert,01033,"POLYGON ((-88.13999 34.5817, -88.13925 34.5878...",01,AL,ALABAMA,01
...,...,...,...,...,...,...,...
3216,Accomack,51001,"MULTIPOLYGON (((-75.24227 38.02721, -75.29687 ...",51,VA,VIRGINIA,51
3217,Bland,51021,"POLYGON ((-81.2251 37.23487, -81.20477 37.2430...",51,VA,VIRGINIA,51
3218,Buchanan,51027,"POLYGON ((-81.9683 37.5378, -81.92787 37.51212...",51,VA,VIRGINIA,51
3219,Charlotte,51037,"POLYGON ((-78.44332 37.0794, -78.49303 36.8912...",51,VA,VIRGINIA,51


In [14]:
merged_data = merged_data[merged_data['State_Abbreviation'] == 'TX' ]
merged_data = merged_data.drop(['State', 'State_Abbreviation', 'State_name', 'Code'], axis=1)
merged_data

,County,Fips,geometry
525,Ochiltree,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49..."
526,Oldham,48359,"POLYGON ((-103.04238 35.18316, -103.0425 35.21..."
527,Parker,48367,"POLYGON ((-97.54418 32.99418, -97.55037 32.580..."
528,Rains,48379,"POLYGON ((-95.66539 32.96043, -95.63502 32.720..."
529,Randall,48381,"POLYGON ((-102.16884 34.74742, -102.16747 35.1..."
...,...,...,...
3089,Jefferson,48245,"POLYGON ((-94.35416 29.56146, -94.35798 29.887..."
3090,Kaufman,48257,"POLYGON ((-96.52999 32.54528, -96.52312 32.545..."
3118,Panola,48365,"POLYGON ((-94.51143 31.97398, -94.59998 31.973..."
3119,Haskell,48207,"POLYGON ((-99.47244 33.39902, -99.47126 32.957..."


In [15]:
merged_data['Fips'] = merged_data['Fips'].astype(int)

In [16]:
merged_data = pd.merge(
    merged_data,
    df,
    left_on=['County', 'Fips'],
    right_on=['COUNTY_NAME', 'COUNTYFIPS']
)
merged_data = merged_data.drop(['COUNTY_NAME', 'COUNTYFIPS'], axis=1)

merged_data

,County,Fips,geometry,Beneficiary_ID,Assessment_Effective_Date,Submitted_HIPPS_Code,Facility_Internal_ID,Age,Gender,American_Indian_or_Alaska_Native,Asian,Black_or_African_American,Hispanic_or_Latino,Native_Hawiian_or_Pacific_Islander,White,ByDiscipline,Days_Cared_For,Agency_Medicare_Number,Primary_Diagnosis_ICD_10_C_M_Code,Other_Diagnosis_Code_1_ICD_10_C_M,Other_Diagnosis_Code_2_ICD_10_C_M,Other_Diagnosis_Code_3_ICD_10_C_M,Other_Diagnosis_Code_4_ICD_10_C_M,Other_Diagnosis_Code_5_ICD_10_C_M,has_diabetes,has_heart_failure,has_hypertension,ever_deceased,ever_readmitted,BMI,BMI_Category,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,ACS_PCT_BACHELOR_DGR,ACS_PCT_COLLEGE_ASSOCIATE_DGR,ACS_PCT_GRADUATE_DGR,ACS_PCT_HS_GRADUATE,ACS_PCT_LT_HS,ACS_PCT_NO_WORK_NO_SCHL_16_19,ACS_PCT_POSTHS_ED,ACS_PCT_VET_BACHELOR,ACS_PCT_VET_COLLEGE,ACS_PCT_VET_HS,ACS_PCT_HH_LIMIT_ENGLISH,ACS_PCT_HH_BROADBAND,ACS_PCT_HH_BROADBAND_ONLY,ACS_PCT_HH_CELLULAR,ACS_PCT_HH_CELLULAR_ONLY,ACS_PCT_HH_DIAL_INTERNET_ONLY,ACS_PCT_HH_INTERNET,ACS_PCT_HH_INTERNET_NO_SUBS,ACS_PCT_HH_NO_COMP_DEV,ACS_PCT_HH_NO_INTERNET,ACS_PCT_HH_OTHER_COMP,ACS_PCT_HH_OTHER_COMP_ONLY,ACS_PCT_HH_PC,ACS_PCT_HH_PC_ONLY,ACS_PCT_HH_SAT_INTERNET,ACS_PCT_HH_SMARTPHONE,ACS_PCT_HH_SMARTPHONE_ONLY,ACS_PCT_HH_TABLET,ACS_PCT_HH_TABLET_ONLY,ACS_PCT_CHILDREN_GRANDPARENT,ACS_PCT_CHILD_1FAM,ACS_PCT_GRANDP_NO_RESPS,ACS_PCT_GRANDP_RESPS_NO_P,ACS_PCT_GRANDP_RESPS_P,ACS_PCT_HH_1PERS,ACS_PCT_HH_ABOVE65,ACS_PCT_HH_ALONE_ABOVE65,ACS_PCT_HH_KID_1PRNT,ACS_TOT_GRANDCHILDREN_GP,ACS_PCT_HEALTH_INC_138_199,ACS_PCT_HEALTH_INC_200_399,ACS_PCT_HEALTH_INC_ABOVE400,ACS_PCT_HEALTH_INC_BELOW137,ACS_PCT_HH_1FAM_FOOD_STMP,ACS_PCT_HH_FOOD_STMP_BLW_POV,ACS_PCT_HH_NO_FD_STMP_BLW_POV,ACS_PCT_HH_PUB_ASSIST,ACS_PCT_INC50,ACS_PCT_INC50_ABOVE65,ACS_PCT_NONVET_POV_18_64,ACS_PCT_PERSON_INC_100_124,ACS_PCT_PERSON_INC_125_199,ACS_PCT_PERSON_INC_ABOVE200,ACS_PCT_PERSON_INC_BELOW99,ACS_PCT_POV_AIAN,ACS_PCT_POV_ASIAN,ACS_PCT_POV_BLACK,ACS_PCT_POV_HISPANIC,ACS_PCT_POV_MULTI,ACS_PCT_POV_NHPI,ACS_PCT_POV_OTHER,ACS_PCT_POV_WHITE,ACS_PCT_VET_POV_18_64,ACS_TOT_POP_POV,ICD_Section,ICD_Range
0,Ochiltree,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",GGGGGGGUzeSUD9e,2017-06-20,1AFKS,9645,90,Female,0,0,0,0,0,1,RN,112,457640,I50.9,M19.90,I10.,E27.1,Q20.8,M17.9,0,1,0,1,0,28.081267,Overweight,8391.0,83.78,1624.0,16.22,15.910000,23.463333,3.603333,34.100000,22.926667,0.000,42.976667,17.953333,46.540000,35.506667,8.370,55.280000,8.616667,73.020000,19.376667,0.036667,82.113333,3.976667,7.206667,13.910000,2.193333,0.000000,77.963333,5.146667,13.303333,80.883333,6.773333,61.783333,1.290000,1.880000,19.400000,6.190000,16.190000,10.953333,24.010000,24.280000,8.576667,15.466667,23.333333,14.536667,31.543333,25.836667,28.086667,48.253333,4.780000,6.543333,12.360000,2.853333,1.640000,11.696667,10.650000,19.216667,57.376667,12.753333,0.000000,0.000,0.000000,11.973333,17.173333,0.000000,8.830000,7.813333,0.000000,3272.000000,"{'Diseases of the circulatory system', 'Endocr...","{'E00-E89', 'I00-I99', 'M00-M99', 'Q00-Q99'}"
1,Ochiltree,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",GGGGGGGUe9ezeDN,2017-12-20,1CGPT,9645,82,Male,0,0,0,0,0,1,RN,0,457640,S72.001S,Z51.89,R53.1,I50.9,M19.90,I10.,0,1,1,0,0,43.066384,Obese-Class3,8391.0,83.78,1624.0,16.22,15.910000,23.463333,3.603333,34.100000,22.926667,0.000,42.976667,17.953333,46.540000,35.506667,8.370,55.280000,8.616667,73.020000,19.376667,0.036667,82.113333,3.976667,7.206667,13.910000,2.193333,0.000000,77.963333,5.146667,13.303333,80.883333,6.773333,61.783333,1.290000,1.880000,19.400000,6.190000,16.190000,10.953333,24.010000,24.280000,8.576667,15.466667,23.333333,14.536667,31.543333,25.836667,28.086667,48.253333,4.780000,6.543333,12.360000,2.853333,1.640000,11.696667,10.650000,19.216667,57.376667,12.753333,0.000000,0.000,0.000000,11.973333,17.173333,0.000000,8.830000,7.813333,0.000000,3272.000000,"{'Diseases of the circulatory system', 'poison...","{'M00-M99', 'S00-T88', 'Z00-Z99', 'I00-I99', '..."
2,Ochiltree

In [21]:
merged_data.columns.tolist()

['County',
 'Fips',
 'geometry',
 'Beneficiary_ID',
 'Assessment_Effective_Date',
 'Submitted_HIPPS_Code',
 'Facility_Internal_ID',
 'Age',
 'Gender',
 'American_Indian_or_Alaska_Native',
 'Asian',
 'Black_or_African_American',
 'Hispanic_or_Latino',
 'Native_Hawiian_or_Pacific_Islander',
 'White',
 'ByDiscipline',
 'Days_Cared_For',
 'Agency_Medicare_Number',
 'Primary_Diagnosis_ICD_10_C_M_Code',
 'Other_Diagnosis_Code_1_ICD_10_C_M',
 'Other_Diagnosis_Code_2_ICD_10_C_M',
 'Other_Diagnosis_Code_3_ICD_10_C_M',
 'Other_Diagnosis_Code_4_ICD_10_C_M',
 'Other_Diagnosis_Code_5_ICD_10_C_M',
 'has_diabetes',
 'has_heart_failure',
 'has_hypertension',
 'ever_deceased',
 'ever_readmitted',
 'BMI',
 'BMI_Category',
 'POP_URB',
 'POPPCT_URB',
 'POP_RUR',
 'POPPCT_RUR',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
 'ACS_PCT_GRADUATE_DGR',
 'ACS_PCT_HS_GRADUATE',
 'ACS_PCT_LT_HS',
 'ACS_PCT_NO_WORK_NO_SCHL_16_19',
 'ACS_PCT_POSTHS_ED',
 'ACS_PCT_VET_BACHELOR',
 'ACS_PCT_VET_COLLEGE',
 

In [18]:
merged_data.to_csv('data/texasCountySummariesOASIS.csv', index=False)

In [19]:
merged_data_copy1 = merged_data.copy()

In [20]:
merged_data_copy2 =merged_data_copy1[['Beneficiary_ID', 'County', 'ever_readmitted',
 'Fips',
 'geometry',
 'Submitted_HIPPS_Code',
 'Facility_Internal_ID',
 'Age',
 'Gender',
 'American_Indian_or_Alaska_Native',
 'Asian',
 'Black_or_African_American',
 'Hispanic_or_Latino',
 'Native_Hawiian_or_Pacific_Islander',
 'White',
 'ByDiscipline',
 'Days_Cared_For',
 'Agency_Medicare_Number',
 'Primary_Diagnosis_ICD_10_C_M_Code',
 'Other_Diagnosis_Code_1_ICD_10_C_M',
 'Other_Diagnosis_Code_2_ICD_10_C_M',
 'Other_Diagnosis_Code_3_ICD_10_C_M',
 'Other_Diagnosis_Code_4_ICD_10_C_M',
 'Other_Diagnosis_Code_5_ICD_10_C_M',
 'has_diabetes',
 'has_heart_failure',
 'has_hypertension',
 'ever_deceased',
 'BMI',
 'BMI_Category',
 'POP_URB',
 'POPPCT_URB',
 'POP_RUR',
 'POPPCT_RUR',
 'ICD_Section',
 'ICD_Range']]

In [22]:
merged_data_copy2

,Beneficiary_ID,County,ever_readmitted,Fips,geometry,Submitted_HIPPS_Code,Facility_Internal_ID,Age,Gender,American_Indian_or_Alaska_Native,Asian,Black_or_African_American,Hispanic_or_Latino,Native_Hawiian_or_Pacific_Islander,White,ByDiscipline,Days_Cared_For,Agency_Medicare_Number,Primary_Diagnosis_ICD_10_C_M_Code,Other_Diagnosis_Code_1_ICD_10_C_M,Other_Diagnosis_Code_2_ICD_10_C_M,Other_Diagnosis_Code_3_ICD_10_C_M,Other_Diagnosis_Code_4_ICD_10_C_M,Other_Diagnosis_Code_5_ICD_10_C_M,has_diabetes,has_heart_failure,has_hypertension,ever_deceased,BMI,BMI_Category,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,ICD_Section,ICD_Range
0,GGGGGGGUzeSUD9e,Ochiltree,0,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",1AFKS,9645,90,Female,0,0,0,0,0,1,RN,112,457640,I50.9,M19.90,I10.,E27.1,Q20.8,M17.9,0,1,0,1,28.081267,Overweight,8391.0,83.78,1624.0,16.22,"{'Diseases of the circulatory system', 'Endocr...","{'E00-E89', 'I00-I99', 'M00-M99', 'Q00-Q99'}"
1,GGGGGGGUe9ezeDN,Ochiltree,0,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",1CGPT,9645,82,Male,0,0,0,0,0,1,RN,0,457640,S72.001S,Z51.89,R53.1,I50.9,M19.90,I10.,0,1,1,0,43.066384,Obese-Class3,8391.0,83.78,1624.0,16.22,"{'Diseases of the circulatory system', 'poison...","{'M00-M99', 'S00-T88', 'Z00-Z99', 'I00-I99', '..."
2,GGGGGGGWGXU9DSW,Ochiltree,0,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",1BGKS,9645,63,Female,0,0,0,0,0,1,RN,0,457640,I50.9,J44.9,Z51.89,J18.0,I10.,E11.9,0,1,0,0,67.849376,Obese-Class3,8391.0,83.78,1624.0,16.22,"{'Diseases of the circulatory system', 'Endocr...","{'J00-J99', 'I00-I99', 'Z00-Z99', 'E00-E89'}"
3,GGGGGGGUDGzWXUU,Ochiltree,0,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",1AGPS,9645,85,Male,0,0,0,0,0,1,RN,41,457640,I25.10,I21.3,R53.1,Z51.89,T14.8,Z48.00,0,0,0,1,27.061851,Overweight,8391.0,83.78,1624.0,16.22,"{'Diseases of the circulatory system', 'poison...","{'R00-R99', 'I00-I99', 'Z00-Z99', 'S00-T88'}"
4,GGGGGGU9WU9eX9z,Ochiltree,0,48357,"POLYGON ((-100.93606 36.4996, -100.91851 36.49...",1CGPS,9645,68,Male,0,0,0,1,0,0,RN,0,457640,I63.9,G81.91,Z51.89,E10.9,I10.,NaN,0,0,0,0,30.782249,Obese-Class1,8391.0,83.78,1624.0,16.22,"{'Diseases of the nervous system', 'Diseases o...","{'G00-G99', 'I00-I99', 'Z00-Z99', 'E00-E89'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221952,GGGGGGGWGe99NUe,Potter,1,48375,"POLYGON ((-101.62294 35.18312, -102.16747 35.1...",1CHKS,9626,87,Male,0,0,0,1,0,0,RN,47,457754,F03.90,I12.9,N18.3,M19.90,M25.562,N40.0,0,0,0,1,32.612544,Obese-Class1,105174.0,88.74,13351.0,11.26,"{'Diseases of the circulatory system', 'Behavi...","{'I00-I99', 'M00-M99', 'N00-N99', 'F01-F99'}"
221953,GGGGGGNGNUeNGe9,Potter,0,48375,"POLYGON ((-101.62294 35.18312, -102.16747 35.1...",2AFKS,366724,63,Female,0,0,0,0,0,1,RN,55,677943,R07.81,M25.572,W01.190D,M06.9,I10.,F32.9,0,0,0,1,38.616943,Obese-Class2,105174.0,88.74,13351.0,11.26,"{'Diseases of the circulatory system', 'Behavi...","{'M00-M99', 'F01-F99', 'V00-Y99', 'I00-I99', '..."
221954,GGGGGGUU9zDSzSS,Potter,1,48375,"POLYGON ((-101.62294 35.18312, -102.16747 35.1...",5AHKS,9644,68,Female,0,0,0,0,0,1,RN,127,459482,R41.3,F33.2,G20.,F41.3,R53.83,F99.,0,0,0,1,24.028320,Normal weight,105174.0,88.74,13351.0,11.26,"{'Diseases of the nervous system', 'Behavioral...","{'F01-F99', 'G00-G99', 'R00-R99'}"
221955,GGGGGGXzG9G9Uze,Potter,0,48375,"POLYGON ((-101.62294 35.18312, -102.16747 35.1...",2BHMS,9644,71,Female,0,0,0,0,0,1,RN,0,459482,N39.0,M54.5,J44.9,F17.200,I12.9,N18.3,0,0,0,0,31.928783,Obese-Class1,105174.0,88.74,13351.0,11.26,"{'Diseases of the circulatory system', 'Behavi...","{'M00-M99', 'F01-F99', 'N00-N99', 'I00-I99', '..."


In [23]:
# Grouping the data
grouped_df = merged_data_copy2.groupby(['County', 'Fips']).agg(
    BENE_ID_unique=('Beneficiary_ID', 'nunique'),
    FACINTID_unique=('Facility_Internal_ID', 'nunique'),
    SBMHPSCD_unique=('Submitted_HIPPS_Code', 'nunique'),
    Age_mean=('Age', 'mean'),
    Gender_counts=('Gender', lambda x: x.value_counts().to_dict()),
    American_Indian_or_Alaska_Native_unique=('American_Indian_or_Alaska_Native', 'nunique'),
    Black_or_African_American_unique=('Black_or_African_American', 'nunique'),
    Hispanic_or_Latino_unique=('Hispanic_or_Latino', 'nunique'),
    Native_Hawiian_or_Pacific_Islander_unique=('Native_Hawiian_or_Pacific_Islander', 'nunique'),
    Unknown_Race_Ethnicity_unique=('American_Indian_or_Alaska_Native', 'nunique'),    
    White_unique=('White', 'nunique'),    
    ByDiscipline_counts=('ByDiscipline', lambda x: x.value_counts().to_dict()),
    M0010_counts=('Agency_Medicare_Number', lambda x: x.value_counts().to_dict()),
    Days_Cared_For_mean=('Days_Cared_For', 'mean'),
    BMI_Category_counts=('BMI_Category', lambda x: x.value_counts().to_dict()),
    Diabetic_patient_count=('has_diabetes', lambda x: (x == 1).sum()),
    HeartFailure_patient_count=('has_heart_failure', lambda x: (x == 1).sum()),
    Hypertension_patient_count=('has_hypertension', lambda x: (x == 1).sum()),
    Readmission_1_count=('ever_readmitted', lambda x: (x == 1).sum()),
    Readmission_0_count=('ever_readmitted', lambda x: (x == 0).sum()),
    geometry=('geometry', 'first'),
    POP_URB=('POP_URB', 'first'),
    POPPCT_URB=('POPPCT_URB', 'first'),
    POP_RUR=('POP_RUR', 'first'),
    POPPCT_RUR=('POPPCT_RUR', 'first'),
    ICD_Section=('ICD_Section', lambda x: ','.join(sorted(set(','.join(x).split(','))))),
    ICD_Range=('ICD_Range', lambda x: ','.join(sorted(set(','.join(x).split(',')))))
).reset_index()
grouped_df

,County,Fips,BENE_ID_unique,FACINTID_unique,SBMHPSCD_unique,Age_mean,Gender_counts,American_Indian_or_Alaska_Native_unique,Black_or_African_American_unique,Hispanic_or_Latino_unique,Native_Hawiian_or_Pacific_Islander_unique,Unknown_Race_Ethnicity_unique,White_unique,ByDiscipline_counts,M0010_counts,Days_Cared_For_mean,BMI_Category_counts,Diabetic_patient_count,HeartFailure_patient_count,Hypertension_patient_count,Readmission_1_count,Readmission_0_count,geometry,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,ICD_Section,ICD_Range
0,Anderson,48001,676,28,127,75.038462,"{'Female': 394, 'Male': 282}",2,2,2,1,2,2,"{'RN': 652, 'PT': 24}","{'678197': 165, '458346': 83, '679007': 83, '6...",88.400888,"{'Overweight': 212, 'Normal weight': 173, 'Obe...",81,65,185,115,561,"POLYGON ((-95.42851 32.08448, -95.44675 31.843...",18615.0,32.14,39307.0,67.86,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'D50-D89', ..."
1,Andrews,48003,114,10,47,75.877193,"{'Female': 67, 'Male': 47}",1,2,2,1,1,2,{'RN': 114},"{'457596': 71, '453104': 8, '747112': 8, '7477...",88.833333,"{'Overweight': 40, 'Normal weight': 28, 'Obese...",24,23,41,22,92,"POLYGON ((-102.28705 32.08699, -102.79909 32.0...",15201.0,81.68,3409.0,18.32,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'D50-D89', ..."
2,Angelina,48005,997,33,178,73.955868,"{'Female': 622, 'Male': 375}",2,2,2,1,2,2,"{'RN': 996, 'PT': 1}","{'679509': 249, '747328': 156, '679108': 109, ...",93.914744,"{'Overweight': 290, 'Normal weight': 268, 'Obe...",165,70,261,163,834,"POLYGON ((-94.32662 31.22475, -94.12963 31.099...",41551.0,48.09,44844.0,51.91,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'C00-D49'},..."
3,Aransas,48007,349,33,108,74.785100,"{'Male': 186, 'Female': 163}",1,2,2,2,1,2,"{'RN': 342, 'PT': 7}","{'747863': 79, '459319': 36, '679129': 36, '67...",78.719198,"{'Overweight': 109, 'Normal weight': 101, 'Obe...",52,21,89,50,299,"MULTIPOLYGON (((-96.85346 28.06134, -96.92905 ...",18212.0,76.42,5618.0,23.58,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'C00-D49'},..."
4,Archer,48009,126,16,60,75.134921,"{'Female': 76, 'Male': 50}",1,1,2,1,1,2,"{'RN': 124, 'PT': 2}","{'677297': 30, '678210': 18, '743162': 11, '67...",110.492063,"{'Normal weight': 44, 'Overweight': 32, 'Obese...",26,8,32,22,104,"POLYGON ((-98.95309 33.834, -98.74236 33.83455...",1082.0,12.64,7478.0,87.36,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'C00-D49'},..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,Wood,48499,1007,50,174,74.703078,"{'Female': 587, 'Male': 420}",2,2,2,1,2,2,"{'RN': 968, 'PT': 39}","{'677037': 196, '459457': 118, '457507': 109, ...",83.170804,"{'Normal weight': 312, 'Overweight': 261, 'Obe...",117,106,256,156,851,"POLYGON ((-95.15341 32.57012, -95.16752 32.543...",5699.0,12.71,39144.0,87.29,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'C00-D49'},..."
248,Yoakum,48501,50,5,28,76.460000,"{'Female': 26, 'Male': 24}",1,1,2,1,1,2,{'RN': 50},"{'679220': 38, '677566': 9, '458364': 1, '6790...",73.360000,"{'Normal weight': 23, 'Obese-Class1': 11, 'Ove...",6,11,10,10,40,"POLYGON ((-102.59502 32.95883, -103.06466 32.9...",0.0,0.00,7694.0,100.00,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'D50-D89', ..."
249,Young,48503,339,15,109,75.306785,"{'Female': 203, 'Male': 136}",1,2,2,1,1,2,"{'RN': 330, 'PT': 9}","{'677917': 114, '678210': 98, '677065': 66, '6...",120.179941,"{'Normal weight': 118, 'Overweight': 98, 'Obes...",50,22,82,52,287,"POLYGON ((-98.95088 32.95692, -98.95394 33.397...",8585.0,48.05,9282.0,51.95,'Behavioral and Neurodevelopmental disorders'...,"'A00-B99', 'A00-B99'}, 'C00-D49', 'C00-D49'},..."
250,Zapata,48505,96,20,43,71.656250,"{'Female': 54, 'Male': 42}",2,1,2,1,2,2,{'RN': 9

In [24]:
grouped_df.columns

Index(['County', 'Fips', 'BENE_ID_unique', 'FACINTID_unique',
       'SBMHPSCD_unique', 'Age_mean', 'Gender_counts',
       'American_Indian_or_Alaska_Native_unique',
       'Black_or_African_American_unique', 'Hispanic_or_Latino_unique',
       'Native_Hawiian_or_Pacific_Islander_unique',
       'Unknown_Race_Ethnicity_unique', 'White_unique', 'ByDiscipline_counts',
       'M0010_counts', 'Days_Cared_For_mean', 'BMI_Category_counts',
       'Diabetic_patient_count', 'HeartFailure_patient_count',
       'Hypertension_patient_count', 'Readmission_1_count',
       'Readmission_0_count', 'geometry', 'POP_URB', 'POPPCT_URB', 'POP_RUR',
       'POPPCT_RUR', 'ICD_Section', 'ICD_Range'],
      dtype='object')

In [25]:
# Filter Texas counties
texas_counties_geojson = {
    "type": "FeatureCollection",
    "features": [feature for feature in counties['features'] if feature['properties']['STATE'] == '48']
}


In [26]:
grouped_df.to_csv('data/streamlit_df.csv', index=False)

In [27]:
# Grouping the data
grouped_df = merged_data_copy1.groupby(['County', 'Fips']).agg(
    BENE_ID_unique=('Beneficiary_ID', 'nunique'),
    FACINTID_unique=('Facility_Internal_ID', 'nunique'),
    SBMHPSCD_unique=('Submitted_HIPPS_Code', 'nunique'),
    Diabetic_patient_count=('has_diabetes', lambda x: (x == 1).sum()),
    HeartFailure_patient_count=('has_heart_failure', lambda x: (x == 1).sum()),
    Hypertension_patient_count=('has_hypertension', lambda x: (x == 1).sum()),
    Readmission_1_count=('ever_readmitted', lambda x: (x == 1).sum()),
    Readmission_0_count=('ever_readmitted', lambda x: (x == 0).sum()),
    geometry=('geometry', 'first'),
    POP_URB=('POP_URB', 'first'),
    POPPCT_URB=('POPPCT_URB', 'first'),
    POP_RUR=('POP_RUR', 'first'),
    POPPCT_RUR=('POPPCT_RUR', 'first'),
).reset_index()
grouped_df

,County,Fips,BENE_ID_unique,FACINTID_unique,SBMHPSCD_unique,Diabetic_patient_count,HeartFailure_patient_count,Hypertension_patient_count,Readmission_1_count,Readmission_0_count,geometry,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR
0,Anderson,48001,676,28,127,81,65,185,115,561,"POLYGON ((-95.42851 32.08448, -95.44675 31.843...",18615.0,32.14,39307.0,67.86
1,Andrews,48003,114,10,47,24,23,41,22,92,"POLYGON ((-102.28705 32.08699, -102.79909 32.0...",15201.0,81.68,3409.0,18.32
2,Angelina,48005,997,33,178,165,70,261,163,834,"POLYGON ((-94.32662 31.22475, -94.12963 31.099...",41551.0,48.09,44844.0,51.91
3,Aransas,48007,349,33,108,52,21,89,50,299,"MULTIPOLYGON (((-96.85346 28.06134, -96.92905 ...",18212.0,76.42,5618.0,23.58
4,Archer,48009,126,16,60,26,8,32,22,104,"POLYGON ((-98.95309 33.834, -98.74236 33.83455...",1082.0,12.64,7478.0,87.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,Wood,48499,1007,50,174,117,106,256,156,851,"POLYGON ((-95.15341 32.57012, -95.16752 32.543...",5699.0,12.71,39144.0,87.29
248,Yoakum,48501,50,5,28,6,11,10,10,40,"POLYGON ((-102.59502 32.95883, -103.06466 32.9...",0.0,0.00,7694.0,100.00
249,Young,48503,339,15,109,50,22,82,52,287,"POLYGON ((-98.95088 32.95692, -98.95394 33.397...",8585.0,48.05,9282.0,51.95
250,Zapata,48505,96,20,43,25,6,40,21,75,"POLYGON ((-98.95423 26.78569, -99.01111 26.675...",10942.0,78.78,2947.0,21.22


In [28]:
sdoh =merged_data_copy1[['County',
 'Fips',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
 'ACS_PCT_GRADUATE_DGR',
 'ACS_PCT_HS_GRADUATE',
 'ACS_PCT_LT_HS',
 'ACS_PCT_NO_WORK_NO_SCHL_16_19',
 'ACS_PCT_POSTHS_ED',
 'ACS_PCT_VET_BACHELOR',
 'ACS_PCT_VET_COLLEGE',
 'ACS_PCT_VET_HS',
 'ACS_PCT_HH_LIMIT_ENGLISH',
 'ACS_PCT_HH_BROADBAND',
 'ACS_PCT_HH_BROADBAND_ONLY',
 'ACS_PCT_HH_CELLULAR',
 'ACS_PCT_HH_CELLULAR_ONLY',
 'ACS_PCT_HH_DIAL_INTERNET_ONLY',
 'ACS_PCT_HH_INTERNET',
 'ACS_PCT_HH_INTERNET_NO_SUBS',
 'ACS_PCT_HH_NO_COMP_DEV',
 'ACS_PCT_HH_NO_INTERNET',
 'ACS_PCT_HH_OTHER_COMP',
 'ACS_PCT_HH_OTHER_COMP_ONLY',
 'ACS_PCT_HH_PC',
 'ACS_PCT_HH_PC_ONLY',
 'ACS_PCT_HH_SAT_INTERNET',
 'ACS_PCT_HH_SMARTPHONE',
 'ACS_PCT_HH_SMARTPHONE_ONLY',
 'ACS_PCT_HH_TABLET',
 'ACS_PCT_HH_TABLET_ONLY',
 'ACS_PCT_CHILDREN_GRANDPARENT',
 'ACS_PCT_CHILD_1FAM',
 'ACS_PCT_GRANDP_NO_RESPS',
 'ACS_PCT_GRANDP_RESPS_NO_P',
 'ACS_PCT_GRANDP_RESPS_P',
 'ACS_PCT_HH_1PERS',
 'ACS_PCT_HH_ABOVE65',
 'ACS_PCT_HH_ALONE_ABOVE65',
 'ACS_PCT_HH_KID_1PRNT',
 'ACS_TOT_GRANDCHILDREN_GP',
 'ACS_PCT_HEALTH_INC_138_199',
 'ACS_PCT_HEALTH_INC_200_399',
 'ACS_PCT_HEALTH_INC_ABOVE400',
 'ACS_PCT_HEALTH_INC_BELOW137',
 'ACS_PCT_HH_1FAM_FOOD_STMP',
 'ACS_PCT_HH_FOOD_STMP_BLW_POV',
 'ACS_PCT_HH_NO_FD_STMP_BLW_POV',
 'ACS_PCT_HH_PUB_ASSIST',
 'ACS_PCT_INC50',
 'ACS_PCT_INC50_ABOVE65',
 'ACS_PCT_NONVET_POV_18_64',
 'ACS_PCT_PERSON_INC_100_124',
 'ACS_PCT_PERSON_INC_125_199',
 'ACS_PCT_PERSON_INC_ABOVE200',
 'ACS_PCT_PERSON_INC_BELOW99',
 'ACS_PCT_POV_AIAN',
 'ACS_PCT_POV_ASIAN',
 'ACS_PCT_POV_BLACK',
 'ACS_PCT_POV_HISPANIC',
 'ACS_PCT_POV_MULTI',
 'ACS_PCT_POV_NHPI',
 'ACS_PCT_POV_OTHER',
 'ACS_PCT_POV_WHITE',
 'ACS_PCT_VET_POV_18_64',
 'ACS_TOT_POP_POV',
]]

In [29]:
df = grouped_df.merge(sdoh, on=['County', 'Fips']).drop_duplicates()
df

,County,Fips,BENE_ID_unique,FACINTID_unique,SBMHPSCD_unique,Diabetic_patient_count,HeartFailure_patient_count,Hypertension_patient_count,Readmission_1_count,Readmission_0_count,geometry,POP_URB,POPPCT_URB,POP_RUR,POPPCT_RUR,ACS_PCT_BACHELOR_DGR,ACS_PCT_COLLEGE_ASSOCIATE_DGR,ACS_PCT_GRADUATE_DGR,ACS_PCT_HS_GRADUATE,ACS_PCT_LT_HS,ACS_PCT_NO_WORK_NO_SCHL_16_19,ACS_PCT_POSTHS_ED,ACS_PCT_VET_BACHELOR,ACS_PCT_VET_COLLEGE,ACS_PCT_VET_HS,ACS_PCT_HH_LIMIT_ENGLISH,ACS_PCT_HH_BROADBAND,ACS_PCT_HH_BROADBAND_ONLY,ACS_PCT_HH_CELLULAR,ACS_PCT_HH_CELLULAR_ONLY,ACS_PCT_HH_DIAL_INTERNET_ONLY,ACS_PCT_HH_INTERNET,ACS_PCT_HH_INTERNET_NO_SUBS,ACS_PCT_HH_NO_COMP_DEV,ACS_PCT_HH_NO_INTERNET,ACS_PCT_HH_OTHER_COMP,ACS_PCT_HH_OTHER_COMP_ONLY,ACS_PCT_HH_PC,ACS_PCT_HH_PC_ONLY,ACS_PCT_HH_SAT_INTERNET,ACS_PCT_HH_SMARTPHONE,ACS_PCT_HH_SMARTPHONE_ONLY,ACS_PCT_HH_TABLET,ACS_PCT_HH_TABLET_ONLY,ACS_PCT_CHILDREN_GRANDPARENT,ACS_PCT_CHILD_1FAM,ACS_PCT_GRANDP_NO_RESPS,ACS_PCT_GRANDP_RESPS_NO_P,ACS_PCT_GRANDP_RESPS_P,ACS_PCT_HH_1PERS,ACS_PCT_HH_ABOVE65,ACS_PCT_HH_ALONE_ABOVE65,ACS_PCT_HH_KID_1PRNT,ACS_TOT_GRANDCHILDREN_GP,ACS_PCT_HEALTH_INC_138_199,ACS_PCT_HEALTH_INC_200_399,ACS_PCT_HEALTH_INC_ABOVE400,ACS_PCT_HEALTH_INC_BELOW137,ACS_PCT_HH_1FAM_FOOD_STMP,ACS_PCT_HH_FOOD_STMP_BLW_POV,ACS_PCT_HH_NO_FD_STMP_BLW_POV,ACS_PCT_HH_PUB_ASSIST,ACS_PCT_INC50,ACS_PCT_INC50_ABOVE65,ACS_PCT_NONVET_POV_18_64,ACS_PCT_PERSON_INC_100_124,ACS_PCT_PERSON_INC_125_199,ACS_PCT_PERSON_INC_ABOVE200,ACS_PCT_PERSON_INC_BELOW99,ACS_PCT_POV_AIAN,ACS_PCT_POV_ASIAN,ACS_PCT_POV_BLACK,ACS_PCT_POV_HISPANIC,ACS_PCT_POV_MULTI,ACS_PCT_POV_NHPI,ACS_PCT_POV_OTHER,ACS_PCT_POV_WHITE,ACS_PCT_VET_POV_18_64,ACS_TOT_POP_POV
0,Anderson,48001,676,28,127,81,65,185,115,561,"POLYGON ((-95.42851 32.08448, -95.44675 31.843...",18615.0,32.14,39307.0,67.86,7.841667,32.505833,4.401667,38.857500,16.396667,0.484167,44.747500,21.754167,43.486667,34.760000,1.988333,45.035833,12.244167,52.398333,13.515000,0.416667,67.963333,2.157500,11.767500,21.545833,6.010000,0.0000,61.189167,4.223333,16.269167,72.360000,12.733333,46.878333,0.625000,10.084167,30.835000,37.776667,20.465833,33.424167,23.509167,29.446667,10.115833,16.695833,103.250000,19.629167,32.141667,21.355000,26.874167,27.555000,4.674167,6.685833,12.410833,4.845833,2.404167,18.850000,4.595833,21.844167,53.496667,20.064167,18.616667,9.615000,22.032500,21.848333,22.510833,0.0,10.224167,9.229167,2.745000,3700.166667
676,Andrews,48003,114,10,47,24,23,41,22,92,"POLYGON ((-102.28705 32.08699, -102.79909 32.0...",15201.0,81.68,3409.0,18.32,11.447500,27.912500,4.192500,33.095000,23.350000,0.000000,43.550000,28.932500,31.685000,39.382500,8.120000,63.585000,6.152500,80.537500,18.565000,0.127500,89.522500,0.722500,6.565000,9.755000,1.482500,0.2375,70.922500,3.365000,8.900000,85.825000,12.707500,63.812500,0.185000,8.790000,27.960000,47.802500,19.100000,33.097500,17.877500,22.322500,7.277500,16.767500,102.500000,9.615000,35.280000,36.475000,18.632500,11.412500,3.515000,6.657500,7.582500,7.362500,3.050000,9.477500,5.740000,12.320000,71.752500,10.190000,0.000000,0.000000,0.000000,16.182500,2.860000,0.0,6.445000,3.332500,4.860000,4529.500000
790,Angelina,48005,997,33,178,165,70,261,163,834,"POLYGON ((-94.32662 31.22475, -94.12963 31.099...",41551.0,48.09,44844.0,51.91,12.648571,33.956190,5.838095,30.977619,16.579048,4.775238,52.443810,18.338571,41.577619,40.083333,2.256667,63.463333,8.862381,74.194286,14.499524,0.292857,85.059524,1.530000,10.047143,13.410952,2.098571,0.0000,70.207143,5.489524,6.886667,80.573333,10.967143,59.636190,1.046190,12.400476,35.436667,45.828095,21.942857,32.228571,23.362857,31.778095,9.943333,23.423810,120.857143,15.026667,31.463810,25.890952,27.619524,42.563333,9.342381,6.512857,19.982857,6.592381,3.434286,16.405238,7.108571,18.352857,57.371429,17.166190,10.395714,2.931429,14.470000,29.201429,12.452857,0.0,11.297143,14.445238,10.269524,4018.190476
1787,Aransas,48007,349,33,108,52,21,89,50,299,"MULTIPOLYGON (((-96.85346 28.06134, -96.92905 ...",18212.0,76.42,561

In [30]:
print(df['geometry'].dtype)

geometry


In [31]:
topics = {
    'Attainment':['ACS_PCT_BACHELOR_DGR', 'ACS_PCT_COLLEGE_ASSOCIATE_DGR', 'ACS_PCT_GRADUATE_DGR', 'ACS_PCT_HS_GRADUATE', 'ACS_PCT_LT_HS',
                  'ACS_PCT_NO_WORK_NO_SCHL_16_19', 'ACS_PCT_POSTHS_ED', 'ACS_PCT_VET_BACHELOR', 'ACS_PCT_VET_COLLEGE', 'ACS_PCT_VET_HS'],
    'English Fluency':[ 'ACS_PCT_HH_LIMIT_ENGLISH' ],
    'Internet Connectivity'	:['ACS_PCT_HH_BROADBAND', 'ACS_PCT_HH_BROADBAND_ONLY', 'ACS_PCT_HH_CELLULAR', 'ACS_PCT_HH_CELLULAR_ONLY',
                              'ACS_PCT_HH_DIAL_INTERNET_ONLY', 'ACS_PCT_HH_INTERNET', 'ACS_PCT_HH_INTERNET_NO_SUBS', 'ACS_PCT_HH_NO_COMP_DEV', 'ACS_PCT_HH_NO_INTERNET',
                              'ACS_PCT_HH_OTHER_COMP', 'ACS_PCT_HH_OTHER_COMP_ONLY', 'ACS_PCT_HH_PC', 'ACS_PCT_HH_PC_ONLY', 'ACS_PCT_HH_SAT_INTERNET', 'ACS_PCT_HH_SMARTPHONE', 'ACS_PCT_HH_SMARTPHONE_ONLY', 'ACS_PCT_HH_TABLET', 'ACS_PCT_HH_TABLET_ONLY'],
    'Living Conditions'	:['ACS_PCT_CHILDREN_GRANDPARENT', 'ACS_PCT_CHILD_1FAM', 'ACS_PCT_GRANDP_NO_RESPS', 'ACS_PCT_GRANDP_RESPS_NO_P', 'ACS_PCT_GRANDP_RESPS_P', 
                          'ACS_PCT_HH_1PERS', 'ACS_PCT_HH_ABOVE65', 'ACS_PCT_HH_ALONE_ABOVE65', 'ACS_PCT_HH_KID_1PRNT', 'ACS_TOT_GRANDCHILDREN_GP'],
    'Poverty':['ACS_PCT_HEALTH_INC_138_199', 'ACS_PCT_HEALTH_INC_200_399', 'ACS_PCT_HEALTH_INC_ABOVE400', 'ACS_PCT_HEALTH_INC_BELOW137', 'ACS_PCT_HH_1FAM_FOOD_STMP',
                   'ACS_PCT_HH_FOOD_STMP_BLW_POV', 'ACS_PCT_HH_NO_FD_STMP_BLW_POV', 'ACS_PCT_HH_PUB_ASSIST', 'ACS_PCT_INC50', 'ACS_PCT_INC50_ABOVE65', 
                  'ACS_PCT_NONVET_POV_18_64', 'ACS_PCT_PERSON_INC_100_124', 'ACS_PCT_PERSON_INC_125_199', 'ACS_PCT_PERSON_INC_ABOVE200', 'ACS_PCT_PERSON_INC_BELOW99', 
                  'ACS_PCT_POV_AIAN', 'ACS_PCT_POV_ASIAN', 'ACS_PCT_POV_BLACK', 'ACS_PCT_POV_HISPANIC', 'ACS_PCT_POV_MULTI', 'ACS_PCT_POV_NHPI', 'ACS_PCT_POV_OTHER', 
                  'ACS_PCT_POV_WHITE', 'ACS_PCT_VET_POV_18_64'],
}

In [32]:
disparity_results = []

# Iterate through topics and their indicators
for topic, indicators in topics.items():
    for indicator in indicators:
        avg_value = df[indicator].mean()

        for _, row in df.iterrows():
            # Absolute Index of Disparity
            aid = abs(row[indicator] - avg_value)
            
            # Relative Index of Disparity
            rid = aid / avg_value if avg_value != 0 else 0
            
            # Population-Weighted Index of Disparity
            # Using urban + rural proportion as weights
            total_pop = row['POPPCT_URB'] + row['POPPCT_RUR']
            pwid = aid * total_pop / df[['POPPCT_URB', 'POPPCT_RUR']].sum().sum()
            
            # Population Attributable Proportion (urban vs rural)
            urban_disparity = aid * row['POPPCT_URB']
            rural_disparity = aid * row['POPPCT_RUR']
            total_disparity = urban_disparity + rural_disparity
            pap_urban = urban_disparity / total_disparity if total_disparity != 0 else 0
            pap_rural = rural_disparity / total_disparity if total_disparity != 0 else 0
            
            disparity_results.append({
                'County': row['County'],
                'Topic': topic,
                'Indicator': indicator,
                'AbsoluteIndexDisparity': aid,
                'RelativeIndexDisparity': rid,
                'PopulationWeightedIndexDisparity': pwid,
                'PAP_Urban': pap_urban,
                'PAP_Rural': pap_rural
            })

disparity_df = pd.DataFrame(disparity_results)
disparity_df

,County,Topic,Indicator,AbsoluteIndexDisparity,RelativeIndexDisparity,PopulationWeightedIndexDisparity,PAP_Urban,PAP_Rural
0,Anderson,Attainment,ACS_PCT_BACHELOR_DGR,5.852780,0.427383,0.023225,0.3214,0.6786
1,Andrews,Attainment,ACS_PCT_BACHELOR_DGR,2.246947,0.164077,0.008916,0.8168,0.1832
2,Angelina,Attainment,ACS_PCT_BACHELOR_DGR,1.045875,0.076372,0.004150,0.4809,0.5191
3,Aransas,Attainment,ACS_PCT_BACHELOR_DGR,1.553735,0.113457,0.006166,0.7642,0.2358
4,Archer,Attainment,ACS_PCT_BACHELOR_DGR,0.468886,0.034239,0.001861,0.1264,0.8736
...,...,...,...,...,...,...,...,...
15871,Wood,Poverty,ACS_PCT_VET_POV_18_64,3.069582,0.328869,0.012181,0.1271,0.8729
15872,Yoakum,Poverty,ACS_PCT_VET_POV_18_64,9.333751,1.000000,0.037039,0.0000,1.0000
15873,Young,Poverty,ACS_PCT_VET_POV_18_64,4.743751,0.508236,0.018824,0.4805,0.5195
15874,Zapata,Poverty,ACS_PCT_VET_POV_18_64,9.333751,1.000000,0.037039,0.7878,0.2122


In [33]:
disparity_df['Indicator'] = disparity_df['Indicator'].str.replace('ACS_PCT', '%')
disparity_df['Indicator'] = disparity_df['Indicator'].str.replace('ACS_TOT', 'TOTAL').str.replace('_', ' ')

disparity_df

,County,Topic,Indicator,AbsoluteIndexDisparity,RelativeIndexDisparity,PopulationWeightedIndexDisparity,PAP_Urban,PAP_Rural
0,Anderson,Attainment,% BACHELOR DGR,5.852780,0.427383,0.023225,0.3214,0.6786
1,Andrews,Attainment,% BACHELOR DGR,2.246947,0.164077,0.008916,0.8168,0.1832
2,Angelina,Attainment,% BACHELOR DGR,1.045875,0.076372,0.004150,0.4809,0.5191
3,Aransas,Attainment,% BACHELOR DGR,1.553735,0.113457,0.006166,0.7642,0.2358
4,Archer,Attainment,% BACHELOR DGR,0.468886,0.034239,0.001861,0.1264,0.8736
...,...,...,...,...,...,...,...,...
15871,Wood,Poverty,% VET POV 18 64,3.069582,0.328869,0.012181,0.1271,0.8729
15872,Yoakum,Poverty,% VET POV 18 64,9.333751,1.000000,0.037039,0.0000,1.0000
15873,Young,Poverty,% VET POV 18 64,4.743751,0.508236,0.018824,0.4805,0.5195
15874,Zapata,Poverty,% VET POV 18 64,9.333751,1.000000,0.037039,0.7878,0.2122


In [34]:
disparity_df['Indicator'].unique()

array(['% BACHELOR DGR', '% COLLEGE ASSOCIATE DGR', '% GRADUATE DGR',
       '% HS GRADUATE', '% LT HS', '% NO WORK NO SCHL 16 19',
       '% POSTHS ED', '% VET BACHELOR', '% VET COLLEGE', '% VET HS',
       '% HH LIMIT ENGLISH', '% HH BROADBAND', '% HH BROADBAND ONLY',
       '% HH CELLULAR', '% HH CELLULAR ONLY', '% HH DIAL INTERNET ONLY',
       '% HH INTERNET', '% HH INTERNET NO SUBS', '% HH NO COMP DEV',
       '% HH NO INTERNET', '% HH OTHER COMP', '% HH OTHER COMP ONLY',
       '% HH PC', '% HH PC ONLY', '% HH SAT INTERNET', '% HH SMARTPHONE',
       '% HH SMARTPHONE ONLY', '% HH TABLET', '% HH TABLET ONLY',
       '% CHILDREN GRANDPARENT', '% CHILD 1FAM', '% GRANDP NO RESPS',
       '% GRANDP RESPS NO P', '% GRANDP RESPS P', '% HH 1PERS',
       '% HH ABOVE65', '% HH ALONE ABOVE65', '% HH KID 1PRNT',
       'TOTAL GRANDCHILDREN GP', '% HEALTH INC 138 199',
       '% HEALTH INC 200 399', '% HEALTH INC ABOVE400',
       '% HEALTH INC BELOW137', '% HH 1FAM FOOD STMP',
       '% HH

In [35]:
disparity_df.to_csv('data/streamlit_sdoh.csv', index=False)